In [ ]:
from mushroom_rl.environments import LQR
from mushroom_rl.solvers.lqr import *

STATE_DIM,A_DIM = 3,2
env = LQR.generate(s_dim=STATE_DIM,a_dim=A_DIM,gamma=0.99,episodic=True,horizon=500,random_init=True)

K = compute_lqr_feedback_gain(env)
state = env.reset()
reward = compute_lqr_V(state,env,K)
reward

In [8]:
# %%

import os
import wandb
import argparse
import itertools
import numpy as np
import jax
import jax.numpy as jnp
from jaxrl_m.common import CodeTimer
import logging
import envpool
logging.basicConfig(level=logging.CRITICAL)


def get_batch(i,batches):
    return  jax.tree_map(lambda x: x[i], batches)

def body(i,val):
    agent,batches = val
    return (agent.update_critics(get_batch(i,batches)),batches)

def str2bool(v):
    if isinstance(v, bool):
        return v
    if v.lower() in ('yes', 'true', 't', 'y', '1'):
        return True
    elif v.lower() in ('no', 'false', 'f', 'n', '0'):
        return False
    else:
        raise argparse.ArgumentTypeError('Boolean value expected.')
    

def none_or_str(value):
    if value == 'None':
        return None
    return value

# Set env variables
os.environ["WANDB_API_KEY"]="28996bd59f1ba2c5a8c3f2cc23d8673c327ae230"
os.environ['PYTHONHASHSEED'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

##############################
parser = argparse.ArgumentParser()
parser.add_argument('--algo_name', type=str, default='sac', help='the name of the RL algorithm')
parser.add_argument('--seed',type=int,default=42) 
parser.add_argument('--env_name',type=str,default="Ant-v5") 
parser.add_argument('--project_name',type=str,default="delete") 
parser.add_argument('--gamma',type=float,default=0.99)
parser.add_argument('--max_steps',type=int,default=100_000) 
parser.add_argument('--num_rollouts',type=int,default=5) 
parser.add_argument('--num_critics',type=int,default=5)     
parser.add_argument('--adaptive_critics',type=str2bool,default=False) 
parser.add_argument('--discount_entropy',type=str2bool,default=True) 
parser.add_argument('--discount_actor',type=str2bool,default=True)
parser.add_argument('--use_momentum',type=str2bool,default=False) 
parser.add_argument('--max_episode_steps',type=int,default=500) 
parser.add_argument('--entropy_coeff',type=float,default=1.) 
parser.add_argument('--actor_lr',type=float,default=3e-4) 
parser.add_argument('--temp_lr',type=float,default=3e-4) 
parser.add_argument('--healthy_reward',type=float,default=1.) 


#args = parser.parse_args()
args = parser.parse_args(args=[])

from jaxrl_m.onsac import *

hidden_dims = ()
NUM_UPDATES = 1000


class SACAgent(flax.struct.PyTreeNode):
    rng: PRNGKey
    critic: TrainState
    target_critic: TrainState
    actor: TrainState
    temp: TrainState
    config: dict = nonpytree_field()

    #@jax.jit
    def update_critics(agent,batch: Batch):
        
        new_rng, curr_key, next_key = jax.random.split(agent.rng, 3)

        def update_one_critic(critic):
                            
                def critic_loss_fn(critic_params):
                        
                        
                        next_dist = agent.actor(batch['next_observations'])
                        next_actions, next_log_probs = next_dist.sample_and_log_prob(seed=next_key)
                        next_q  = agent.critic(batch['next_observations'], next_actions,params=critic_params)
                        
                        target_q = batch['rewards'] + agent.config['discount'] * batch['masks'] * next_q
                        target_q = target_q - agent.config['discount'] * batch['masks'] * next_log_probs * agent.temp()
                        target_q = jax.lax.stop_gradient(target_q)
                        
                        q = agent.critic(batch['observations'], batch['actions'],params=critic_params)
                        critic_loss = ((target_q-q)**2).mean() 
                        
                        return critic_loss, {
                        'critic_loss': critic_loss,
                        'q1': q.mean(),
                    }  
                
                new_critic, critic_info = critic.apply_loss_fn(loss_fn=critic_loss_fn, has_aux=True)
                
                return new_critic,critic_info


        new_critics,critic_info = jax.vmap(update_one_critic)(agent.critic)
        agent = agent.replace(rng=new_rng,critic=new_critics)
        
        return agent
    
    @jax.jit
    def update_critics_seq(agent,batches,R2):
       
        new_critic_params = agent.critic.params
        ### Reset optimizers 
        new_opt_state = jax.vmap(agent.critic.tx.init)(new_critic_params)
        new_critics = agent.critic.replace(params=new_critic_params,opt_state=new_opt_state)
        agent = agent.replace(critic=new_critics)
        ### Train critic sequentially
        agent,batches = jax.lax.fori_loop(0,2500,body,(agent,batches))
        
        return agent

    @jax.jit
    def update_actor(agent, batch: Batch,R2):

        new_rng, curr_key, next_key = jax.random.split(agent.rng, 3)

        def actor_loss_fn(actor_params,R2):
            observations = jnp.repeat(batch['observations'], 10, axis=0)
            discounts = jnp.repeat(batch['discounts'], 10, axis=0)
            masks = jnp.int32(jnp.repeat(batch['masks'], 10, axis=0))

            dist = agent.actor(observations, params=actor_params)
            actions, log_probs = dist.sample_and_log_prob(seed=curr_key)
            call_one_critic = lambda observations,actions,params: agent.critic(observations,actions,params=params)
            q_all = jax.vmap(call_one_critic,in_axes=(None,None,0))(observations, actions,agent.critic.params)##critic_update_info
            
            q_weights = jax.nn.softmax(R2,axis=0)
            q = jnp.sum(q_weights.reshape(-1,1)*q_all,axis=0)

            
            ### Pad Q and logits because actor buffer is padded ###
            q = masks *q
            log_probs = masks * log_probs
            
            if agent.config['discount_actor']:
                actor_loss = (discounts*(log_probs * agent.temp() - q)).sum()/(discounts.sum())
            else :
                actor_loss = (log_probs * agent.temp() - q).sum()/(masks.sum())
            
            if agent.config['discount_entropy']:
                entropy = -1 * (discounts*log_probs).sum()/(discounts.sum())
            else : 
                entropy = -1 * log_probs.sum()/(masks.sum())
            
            return actor_loss, {
                'actor_loss': actor_loss,
                'entropy': entropy,
            }
        
        
        def temp_loss_fn(temp_params, entropy, target_entropy):
            temperature = agent.temp(params=temp_params)
            entropy_diff = entropy-target_entropy
            temp_loss = (temperature * entropy_diff).mean()
            return temp_loss, {
                'temp_loss': temp_loss,
                'temperature': temperature,
                'entropy_diff': entropy_diff,
            }

        
        new_actor, actor_info = agent.actor.apply_loss_fn(actor_loss_fn,True,R2)
        new_temp, temp_info = agent.temp.apply_loss_fn(temp_loss_fn,True,actor_info['entropy'], agent.config['target_entropy'])
        new_temp.params["log_temp"]=jnp.clip(new_temp.params["log_temp"],1e-6,1)
        
        grads,info = jax.grad(actor_loss_fn,has_aux=True)(agent.actor.params,R2)
        
        return agent.replace(rng=new_rng, actor=new_actor,temp=new_temp), {**actor_info,**temp_info},grads
        
        #return agent.replace(rng=new_rng, actor=new_actor,temp=new_temp), {**actor_info,**temp_info}
        
    
    @jax.jit
    def sample_actions(agent,   
                       observations: np.ndarray,
                       seed: PRNGKey,
                       temperature: float = 1.0,
                       ) -> jnp.ndarray:
        
        ### random always true
        actions = agent.actor(observations, temperature=temperature).sample(seed=seed)
        
        return actions



def create_learner(
                seed: int,
                observations: jnp.ndarray,
                actions: jnp.ndarray,
                discount: float,
                num_critics: int,
                discount_actor ,
                discount_entropy,
                adaptive_critics,
                entropy_coeff,
                use_momentum,
                
                actor_lr: float = 3e-4,
                critic_lr: float = 3e-4,
                temp_lr: float =1e-3,## Test
                hidden_dims: Sequence[int] = (256, 256),
                target_entropy: float = None,
            **kwargs):

        print('Extra kwargs:', kwargs)

        rng = jax.random.PRNGKey(seed)
        rng, actor_key, critic_key = jax.random.split(rng, 3)

        action_dim = actions.shape[-1]
        actor_def = Policy(hidden_dims, action_dim=action_dim,
            state_dependent_std=True, tanh_squash_distribution=False,use_bias=False)

        critic_def = OriginalCritic(hidden_dims)
        critic_keys  = jax.random.split(critic_key, num_critics)
        critic_params = jax.vmap(critic_def.init,in_axes=(0,None,None))(critic_keys, observations, actions)['params']
        critics = jax.vmap(TrainState.create,in_axes=(None,0,None))(critic_def,critic_params,optax.adam(learning_rate=critic_lr))

        actor_params = actor_def.init(actor_key, observations)['params']
        temp_def = Temperature()
        temp_params = temp_def.init(rng)['params']
        
        
        if use_momentum:
            temp = TrainState.create(temp_def, temp_params, tx=optax.adam(learning_rate=temp_lr))
            actor = TrainState.create(actor_def, actor_params, tx=optax.adam(learning_rate=actor_lr))
            
        else:
            temp = TrainState.create(temp_def, temp_params, tx=optax.adam(learning_rate=temp_lr,b1=0,b2=0.9))
            actor = TrainState.create(actor_def, actor_params, tx=optax.adam(learning_rate=actor_lr,b1=0,b2=0.9))
            
        if target_entropy is None:

            target_entropy = -entropy_coeff*action_dim

        config = flax.core.FrozenDict(dict(
            discount=discount,
            target_entropy=target_entropy,
            observations=observations,
            actions=actions,  
            num_critics = num_critics, 
            discount_actor = discount_actor, 
            discount_entropy = discount_entropy,
            adaptive_critics = adaptive_critics,
            
        ))

        return SACAgent(rng, critic=critics, target_critic=critics, actor=actor, temp=temp, config=config)



def train(args):
    
    import os
    from functools import partial
    import numpy as np
    import jax
    import tqdm
    import gymnasium as gym


    from jaxrl_m.wandb import setup_wandb, default_wandb_config, get_flag_dict
    import wandb
    from jaxrl_m.evaluation import supply_rng, evaluate, flatten, EpisodeMonitor
    from jaxrl_m.dataset import ReplayBuffer,ActorReplayBuffer
    from collections import deque
    from jax import config
    from jaxrl_m.utils import flatten_rollouts
    from jaxrl_m.evaluate_critic import evaluate_many_critics
    from jaxrl_m.rollout import rollout_policy_lqr
    from jax import config
    config.update("jax_debug_nans", True)

    eval_episodes=10
    batch_size = 256
    max_steps = args.max_steps
    start_steps = 0
    log_interval = 10000
    n_grads = 0

    wandb_config = {
        'project': args.project_name,
        'name':None,
        'hyperparam_dict':args.__dict__,
        }
    #wandb_run = setup_wandb(**wandb_config)



    observation = jnp.ones(env._mdp_info.observation_space.shape)
    action = jnp.ones(env._mdp_info.action_space.shape)
    example_transition = dict(

        observations=observation,
        actions=action,
        rewards=0.0,
        masks=1.0,
        #next_observations=env.observation_space.sample(),
        next_observations=observation,
        discounts=1.0,
    )

    replay_buffer = ReplayBuffer.create(example_transition, size=int(1e5))
    actor_buffer = ActorReplayBuffer.create(example_transition, size=int(args.num_rollouts*args.max_episode_steps))

    agent = create_learner(args.seed,
                        
                    observations=example_transition['observations'][None],
                    actions =example_transition['actions'][None],
                    max_steps=max_steps,
                    discount=args.gamma,
                    discount_actor=args.discount_actor,
                    discount_entropy=args.discount_entropy,
                    adaptive_critics=args.adaptive_critics,
                    num_critics= args.num_critics,
                    entropy_coeff=args.entropy_coeff,
                    temp_lr=args.temp_lr,
                    actor_lr=args.actor_lr,
                    use_momentum=args.use_momentum,
                    hidden_dims=hidden_dims,
                    #**FLAGS.config
                    )

    #K = compute_lqr_feedback_gain(env).reshape((2,2))
    #K = np.zeros((STATE_DIM,A_DIM))
    #print(K,K.shape)
    # actor = agent.actor
    # new_params = actor.params.copy()
    # new_params['means']['kernel'] = jnp.array(K)
    
    # new_actor = actor.replace(params=new_params)
    # agent.replace(actor=new_actor)
    
    exploration_metrics = dict()
    #obs,info = env.reset()    
    exploration_rng = jax.random.PRNGKey(0)
    i = 0
    unlogged_steps,cached_steps = 0,0
    policy_rollouts = deque([], maxlen=20)
    warmup = True
    R2,bias = jnp.ones(args.num_critics),jnp.zeros(args.num_critics)

    
    with tqdm.tqdm(total=max_steps) as pbar:
        
        while (i < max_steps):
            with jax.log_compiles(False):
                warmup=(i < start_steps)
                
                logging.debug('policy rollout')
                replay_buffer,actor_buffer,policy_rollout,policy_return,variance,undisc_policy_return,num_steps = rollout_policy_lqr(
                                                                        agent,env,exploration_rng,
                                                                        replay_buffer,actor_buffer,warmup=warmup,
                                                                        num_rollouts=args.num_rollouts,discount = args.gamma,max_length=args.max_episode_steps)
                
                print(f'policy_return: {policy_return}, undisc_policy_return {undisc_policy_return}')                                                              
                if not warmup : policy_rollouts.append(policy_rollout)
                unlogged_steps += num_steps
                cached_steps += num_steps
                i+=num_steps
                pbar.update(int(num_steps))
                
                if replay_buffer.size > start_steps:
                
                    ### Update critics ###:
                    logging.debug('update critics')
                    transitions = replay_buffer.get_all()
                    idxs = jax.random.choice(agent.rng,a=transitions['observations'].shape[0], shape=(NUM_UPDATES,256), replace=True)
                    batches = jax.vmap(lambda i: jax.tree_map(lambda x: x[i], transitions))(idxs)
                    agent = agent.update_critics_seq(batches,R2)
                    
                  
                   
                    
                    ### Update actor ###
                    actor_batch = actor_buffer.get_all()    
                    agent, actor_update_info,grads = agent.update_actor(actor_batch,R2)    
                    critic_update_info = {}
                    update_info = {**critic_update_info, **actor_update_info}
                    n_grads += 1
                    
                    ### Grad stuff ###
                    def flatten(grads):    
                        tmp = jax.tree_map(lambda x: jnp.reshape(x,(-1,)),grads)
                        tmp = jax.tree_util.tree_flatten(tmp)[0]
                        tmp = jnp.concatenate(tmp)
                        return tmp

                    one = flatten(grads)
                    
                    # print(f'gradient approx {one}')
                    # print('cosine distance',jnp.dot(one.flatten(),two.flatten())/(jnp.linalg.norm(one.flatten())*jnp.linalg.norm(two.flatten())))
                    # wandb.log({'cosine_distance':jnp.dot(one.flatten(),two.flatten())/(jnp.linalg.norm(one.flatten())*jnp.linalg.norm(two.flatten()))}, step=int(i),commit=False)

                
                        
                    
                    ### Log training info ###
                    exploration_metrics = {f'exploration/disc_return': policy_return,'training/std': jnp.sqrt(variance)}
                    train_metrics = {f'training/{k}': v for k, v in update_info.items()}
                    train_metrics['training/undisc_return'] = undisc_policy_return
                                      
                
                    if cached_steps >= int(1e6): 
                        jax.clear_caches()
                        cached_steps = 0
                        print('clearing cache')
            
    #wandb_run.finish()
    return agent,actor_batch

agent,actor_batch = train(args)
#%%


Extra kwargs: {'max_steps': 100000}


  2%|▎         | 2500/100000 [00:00<00:29, 3331.64it/s]

policy_return: -17319.787025365087, undisc_policy_return -446869.2153490729
gradient approx [-3.1043632e-02  3.0619553e-03  5.3539495e-03 -9.2931523e-04
 -2.5145689e-03  1.4259330e-03  1.2888290e+00 -2.6681742e-01
 -9.8418140e-01  2.0374836e-01  2.9614422e-01 -6.1308723e-02]


  5%|▌         | 5000/100000 [00:02<00:41, 2314.40it/s]

policy_return: -8819.127561479298, undisc_policy_return -183844.00354050184
gradient approx [-0.02764252  0.05882753 -0.06941906 -0.00421625 -0.00390947 -0.00284835
  0.69228745 -0.8969477  -0.34233707  0.4435406  -0.8679127   1.124492  ]


...

policy_return: -7821.462333135285, undisc_policy_return -117476.76725321407
gradient approx [-3.6173329e-02  4.0762331e-02  3.4389760e-02  9.8586613e-03
  3.4380055e-03 -3.3542302e-02  1.8081311e+00 -8.6968222e+00
 -6.9490486e-01  3.3423831e+00 -4.3572876e-01  2.0957861e+00]


 15%|█▌        | 15000/100000 [00:06<00:33, 2543.24it/s]

policy_return: -5468.174935260378, undisc_policy_return -84685.86352327796
gradient approx [ 2.3526195e-02  4.0709395e-02 -9.0512363e-03  2.5076088e-01
 -5.7206891e-04  5.1722780e-02  1.4421417e+00 -6.7813525e+00
 -2.5872973e-01  1.2166169e+00  1.8566503e-01 -8.7304825e-01]


 15%|█▌        | 15000/100000 [00:07<00:39, 2137.24it/s]


KeyboardInterrupt: 

In [ ]:
K = compute_lqr_feedback_gain(env)
# agent.actor.params['means']['kernel'] = K
# K = np.array(agent.actor.params['means']['kernel'])

total = 0
gamma = 1
exploration_rng = jax.random.PRNGKey(0)
obs = env.reset()
for i in range(500):

    exploration_rng, key = jax.random.split(exploration_rng)
    action = agent.sample_actions(obs,seed=exploration_rng)
    #action = -K@obs
    
    next_obs, reward, done, info = env.step(action)  
    obs = next_obs

    total+=gamma*reward
    gamma *= 0.99
    

print(total)


In [ ]:
compute_lqr_Q_gaussian_policy_gradient_K(obs,action,env,K,0.01*np.ones_like(K).T)

In [ ]:
K = compute_lqr_feedback_gain(env)
Sigma = np.ones_like(K).T

# K = np.array(agent.actor.params['means']['kernel']).T
# Sigma = np.exp(agent.actor.params["log_stds"]['kernel'])

for i in range(500):
    obs = env.reset()
    compute_lqr_V_gaussian_policy(obs,env,K,Sigma)
#compute_lqr_V(obs,env,K)

In [ ]:
K = compute_lqr_feedback_gain(env)
Sigma = np.ones_like(K).T

# K = np.array(agent.actor.params['means']['kernel']).T
# Sigma = np.exp(agent.actor.params["log_stds"]['kernel'])

for i in range(500):
    obs = env.reset()
    compute_lqr_Q_gaussian_policy_gradient_K(obs,K@obs,env,K,Sigma)
#compute_lqr_V(obs,env,K)